# Notebook 03 - Data Validation & Cleaning
### Spain Tactical Analysis: Qatar 2022 - Euro 2024

**Purpose:** Take the raw event dataset from Notebook 02, unpack coordinates, define pitch zones and progressive actions, standardize player names, and produce a single clean master dataset. Every analytical concept (progressive pass, final third, half-space) is defined here exactly once so that all downstream notebooks comparing 2022 and 2024 use the same logic.

**Outputs:**
- `outputs/data/master_events_cleaned.parquet`

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import ast
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))
from utils.config import OUTPUTS_DATA_DIR

print("Imports OK.")

Imports OK.


---
## Section 1 - Load Raw Events

In [2]:
raw_path = os.path.join(OUTPUTS_DATA_DIR, 'events_raw.parquet')
events = pd.read_parquet(raw_path)

print(f"Loaded {len(events):,} raw events.")
print(f"Columns: {len(events.columns)}")
print(f"Tournaments: {events['tournament'].unique()}")
print(f"Event types: {events['type'].nunique()}")

Loaded 69,945 raw events.
Columns: 114
Tournaments: <ArrowStringArray>
['WC2022', 'EURO2024']
Length: 2, dtype: str
Event types: 33


---
## Section 2 - Unpack Coordinates

StatsBomb stores locations as `[x, y]` lists. These were cast to strings when we saved to parquet. We parse them back and create clean numeric `x`, `y` columns.

**StatsBomb coordinate system:** 120 (length) x 80 (width). The opponent's goal is at x=120, y=40.

In [3]:
def safe_parse(val):
    """Safely parse a stringified list back into a Python list."""
    if pd.isna(val) or val in ('nan', 'None', '', 'NaN'):
        return None
    if isinstance(val, (list, tuple)):
        return val
    if isinstance(val, str):
        try:
            result = ast.literal_eval(val)
            if isinstance(result, (list, tuple)):
                return result
        except (ValueError, SyntaxError):
            pass
    return None

def extract_coord(parsed, index):
    """Extract x (index 0) or y (index 1) from a parsed list."""
    if parsed is not None and len(parsed) > index:
        return float(parsed[index])
    return np.nan

print("Unpacking event locations...")
parsed_loc = events['location'].apply(safe_parse)
events['x'] = parsed_loc.apply(lambda p: extract_coord(p, 0))
events['y'] = parsed_loc.apply(lambda p: extract_coord(p, 1))

print("Unpacking pass end locations...")
if 'pass_end_location' in events.columns:
    parsed_pass = events['pass_end_location'].apply(safe_parse)
    events['pass_end_x'] = parsed_pass.apply(lambda p: extract_coord(p, 0))
    events['pass_end_y'] = parsed_pass.apply(lambda p: extract_coord(p, 1))
else:
    events['pass_end_x'] = np.nan
    events['pass_end_y'] = np.nan

print("Unpacking carry end locations...")
if 'carry_end_location' in events.columns:
    parsed_carry = events['carry_end_location'].apply(safe_parse)
    events['carry_end_x'] = parsed_carry.apply(lambda p: extract_coord(p, 0))
    events['carry_end_y'] = parsed_carry.apply(lambda p: extract_coord(p, 1))
else:
    events['carry_end_x'] = np.nan
    events['carry_end_y'] = np.nan

print("Unpacking shot end locations...")
if 'shot_end_location' in events.columns:
    parsed_shot = events['shot_end_location'].apply(safe_parse)
    events['shot_end_x'] = parsed_shot.apply(lambda p: extract_coord(p, 0))
    events['shot_end_y'] = parsed_shot.apply(lambda p: extract_coord(p, 1))
else:
    events['shot_end_x'] = np.nan
    events['shot_end_y'] = np.nan

has_xy = events['x'].notna().sum()
print(f"\nCoordinate unpacking complete. {has_xy:,} events have x/y coordinates.")

Unpacking event locations...


Unpacking pass end locations...
Unpacking carry end locations...


Unpacking shot end locations...

Coordinate unpacking complete. 69,387 events have x/y coordinates.


---
## Section 3 - Define Pitch Zones

We tag every event with its pitch zone. These definitions are used consistently across all notebooks.

**Pitch thirds (lengthwise):**
- Defensive Third: x < 40
- Middle Third: 40 <= x < 80
- Final Third: x >= 80

**Vertical zones (widthwise):**
- Left Wing: y < 18
- Left Half-space: 18 <= y < 30
- Center: 30 <= y < 50
- Right Half-space: 50 <= y < 62
- Right Wing: y >= 62

In [4]:
def pitch_third(x):
    if pd.isna(x):
        return np.nan
    if x < 40:
        return 'Defensive Third'
    elif x < 80:
        return 'Middle Third'
    else:
        return 'Final Third'

def vertical_zone(y):
    if pd.isna(y):
        return np.nan
    if y < 18:
        return 'Left Wing'
    elif y < 30:
        return 'Left Half-space'
    elif y < 50:
        return 'Center'
    elif y < 62:
        return 'Right Half-space'
    else:
        return 'Right Wing'

events['zone_third']    = events['x'].apply(pitch_third)
events['zone_vertical'] = events['y'].apply(vertical_zone)

events['end_zone_third']    = events['pass_end_x'].apply(pitch_third)
events['end_zone_vertical'] = events['pass_end_y'].apply(vertical_zone)

events['carry_end_zone_third']    = events['carry_end_x'].apply(pitch_third)
events['carry_end_zone_vertical'] = events['carry_end_y'].apply(vertical_zone)

print("Zone distribution (start location):")
print(events[events['team'] == 'Spain']['zone_third'].value_counts())
print("\nZones mapped successfully.")

Zone distribution (start location):


zone_third
Middle Third       13734
Final Third         7602
Defensive Third     5864
Name: count, dtype: int64

Zones mapped successfully.


---
## Section 4 - Define Progressive Actions

**Progressive Pass definition:**
A completed pass that moves the ball at least 25% closer to the center of the opponent's goal (x=120, y=40), OR any completed pass that enters the penalty area from outside it. Passes originating in the team's own deep defensive zone (x < 48) must travel at least 30% closer to count.

**Progressive Carry definition:**
A carry that moves the ball at least 25% closer to the center of the opponent's goal.

In [5]:
def dist_to_goal(x, y):
    return np.sqrt((120.0 - x)**2 + (40.0 - y)**2)

def is_in_penalty_area(x, y):
    return x >= 102 and 18 <= y <= 62

def calc_progressive_pass(row):
    if row['type'] != 'Pass':
        return False
    if pd.notna(row.get('pass_outcome')):
        return False
    x1, y1 = row['x'], row['y']
    x2, y2 = row['pass_end_x'], row['pass_end_y']
    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        return False
    if not is_in_penalty_area(x1, y1) and is_in_penalty_area(x2, y2):
        return True
    d_start = dist_to_goal(x1, y1)
    d_end   = dist_to_goal(x2, y2)
    threshold = 0.70 if x1 < 48 else 0.75
    return d_end <= (threshold * d_start)

def calc_progressive_carry(row):
    if row['type'] != 'Carry':
        return False
    x1, y1 = row['x'], row['y']
    x2, y2 = row['carry_end_x'], row['carry_end_y']
    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        return False
    d_start = dist_to_goal(x1, y1)
    d_end   = dist_to_goal(x2, y2)
    return d_end <= (0.75 * d_start)

print("Calculating progressive passes...")
events['is_progressive_pass'] = events.apply(calc_progressive_pass, axis=1)
print("Calculating progressive carries...")
events['is_progressive_carry'] = events.apply(calc_progressive_carry, axis=1)

spain_events = events[events['team'] == 'Spain']
pp = spain_events['is_progressive_pass'].sum()
pc = spain_events['is_progressive_carry'].sum()
print(f"\nSpain progressive passes  (all matches): {pp:,}")
print(f"Spain progressive carries (all matches): {pc:,}")

Calculating progressive passes...


Calculating progressive carries...



Spain progressive passes  (all matches): 490
Spain progressive carries (all matches): 296


---
## Section 5 - Define Forward Pass Ratio (Directness)

In [6]:
def is_forward_pass(row):
    if row['type'] != 'Pass':
        return np.nan
    if pd.isna(row['x']) or pd.isna(row['pass_end_x']):
        return np.nan
    return row['pass_end_x'] > row['x']

events['is_forward_pass'] = events.apply(is_forward_pass, axis=1)

for t in ['WC2022', 'EURO2024']:
    subset = events[(events['team'] == 'Spain') & (events['tournament'] == t) & (events['type'] == 'Pass')]
    fwd_ratio = subset['is_forward_pass'].mean()
    print(f"{t} -- Spain forward pass ratio: {fwd_ratio:.1%}")

WC2022 -- Spain forward pass ratio: 55.6%
EURO2024 -- Spain forward pass ratio: 58.8%


---
## Section 6 - Player Name Standardization & Aliasing

StatsBomb uses official legal names. 'Pedri' is listed as 'Pedro González López' and 'Nico Williams' is listed as 'Nicholas Williams Arthuer'. 
To make our lives much easier in the downstream notebooks, we will create a `common_name` column that translates these legal names into the football names we all actually use.

In [7]:
# Create a mapping dictionary for Spain's key players
name_aliases = {
    'Pedro González López': 'Pedri',
    'Pablo Martín Páez Gavira': 'Gavi',
    'Rodrigo Hernández Cascante': 'Rodri',
    'Nicholas Williams Arthuer': 'Nico Williams',
    'Lamine Yamal Nasraoui Ebana': 'Lamine Yamal',
    'Sergio Busquets i Burgos': 'Busquets',
    'Jordi Alba Ramos': 'Jordi Alba',
    'Daniel Olmo Carvajal': 'Dani Olmo',
    'Álvaro Borja Morata Martín': 'Morata',
    'Ferrán Torres García': 'Ferran Torres',
    'Aymeric Laporte': 'Laporte',
    'Unai Simón Mendibil': 'Unai Simón',
    'Marc Cucurella Saseta': 'Cucurella',
    'Fabián Ruiz Peña': 'Fabián Ruiz',
    'Alejandro Grimaldo García': 'Grimaldo',
    'Robin Aime Robert Le Normand': 'Le Normand',
    'Martín Zubimendi Ibáñez': 'Zubimendi',
    'Mikel Merino Zazón': 'Merino',
    'Mikel Oyarzabal Ugarte': 'Oyarzabal'
}

def get_common_name(player_name):
    if pd.isna(player_name):
        return np.nan
    # If it's in our mapping, use the alias. Otherwise, keep the original name.
    return name_aliases.get(player_name, player_name)

events['common_name'] = events['player'].apply(get_common_name)

print("Common Name Mapping Results (Key Players):")
print("=" * 50)
for original, common in name_aliases.items():
    # Verify the original name actually exists in the data
    if original in events['player'].values:
        print(f"  [OK] {original[:25]:25s} -> {common}")
    else:
        print(f"  [--] {original[:25]:25s} -> NOT FOUND IN DATA")
print("=" * 50)
print("\nWe will use the 'common_name' column for all visualizations going forward.")

Common Name Mapping Results (Key Players):
  [OK] Pedro González López      -> Pedri
  [OK] Pablo Martín Páez Gavira  -> Gavi
  [OK] Rodrigo Hernández Cascant -> Rodri
  [OK] Nicholas Williams Arthuer -> Nico Williams
  [OK] Lamine Yamal Nasraoui Eba -> Lamine Yamal
  [OK] Sergio Busquets i Burgos  -> Busquets
  [OK] Jordi Alba Ramos          -> Jordi Alba
  [OK] Daniel Olmo Carvajal      -> Dani Olmo
  [OK] Álvaro Borja Morata Martí -> Morata
  [OK] Ferrán Torres García      -> Ferran Torres
  [OK] Aymeric Laporte           -> Laporte
  [OK] Unai Simón Mendibil       -> Unai Simón
  [OK] Marc Cucurella Saseta     -> Cucurella
  [OK] Fabián Ruiz Peña          -> Fabián Ruiz
  [OK] Alejandro Grimaldo García -> Grimaldo
  [OK] Robin Aime Robert Le Norm -> Le Normand
  [OK] Martín Zubimendi Ibáñez   -> Zubimendi
  [OK] Mikel Merino Zazón        -> Merino
  [OK] Mikel Oyarzabal Ugarte    -> Oyarzabal

We will use the 'common_name' column for all visualizations going forward.


---
## Section 7 - Data Quality Checks

In [8]:
print("=" * 60)
print("  DATA QUALITY REPORT")
print("=" * 60)

for etype in ['Pass', 'Shot', 'Carry', 'Pressure', 'Ball Recovery']:
    subset = events[events['type'] == etype]
    missing = subset['x'].isna().sum()
    total = len(subset)
    pct = (missing / total * 100) if total > 0 else 0
    print(f"  {etype:20s} -- {total:,} total, {missing:,} missing coords ({pct:.1f}%)")

spain_passes = events[(events['team'] == 'Spain') & (events['type'] == 'Pass')]
completed = spain_passes['pass_outcome'].isna().sum() 
total_passes = len(spain_passes)
print(f"\nSpain passes: {total_passes:,} total, {completed:,} completed ({completed/total_passes:.1%})")
print("\nData quality checks complete.")

  DATA QUALITY REPORT
  Pass                 -- 20,285 total, 0 missing coords (0.0%)
  Shot                 -- 466 total, 0 missing coords (0.0%)
  Carry                -- 16,472 total, 0 missing coords (0.0%)
  Pressure             -- 5,318 total, 0 missing coords (0.0%)
  Ball Recovery        -- 1,569 total, 0 missing coords (0.0%)

Spain passes: 8,248 total, 7,348 completed (89.1%)

Data quality checks complete.


---
## Section 8 - Clean and Save Master Dataset

In [9]:
cols_to_drop = ['location', 'pass_end_location', 'carry_end_location', 'shot_end_location']
events_clean = events.drop(columns=[c for c in cols_to_drop if c in events.columns])

for col in events_clean.columns:
    if events_clean[col].apply(lambda x: isinstance(x, (list, dict))).any():
        events_clean[col] = events_clean[col].astype(str)

clean_path = os.path.join(OUTPUTS_DATA_DIR, 'master_events_cleaned.parquet')
events_clean.to_parquet(clean_path, index=False)

print(f"Master dataset saved to: {clean_path}")
print(f"Shape: {events_clean.shape[0]:,} rows x {events_clean.shape[1]} columns")

Master dataset saved to: ../outputs/data\master_events_cleaned.parquet
Shape: 69,945 rows x 128 columns


---
## Section 9 - Summary

In [10]:
print("=" * 60)
print("  NOTEBOOK 03 -- COMPLETE")
print("=" * 60)
print(f"  [OK] Coordinates unpacked")
print(f"  [OK] Pitch zones defined")
print(f"  [OK] Progressive metrics computed")
print(f"  [OK] Player names aliased to common names (e.g. Pedri, Gavi)")
print(f"  [OK] Saved: master_events_cleaned.parquet ({events_clean.shape[0]:,} rows)")
print()
print("  NEXT: Notebook 04 -- Spain 2022 Tactical Analysis")
print("=" * 60)

  NOTEBOOK 03 -- COMPLETE
  [OK] Coordinates unpacked
  [OK] Pitch zones defined
  [OK] Progressive metrics computed
  [OK] Player names aliased to common names (e.g. Pedri, Gavi)
  [OK] Saved: master_events_cleaned.parquet (69,945 rows)

  NEXT: Notebook 04 -- Spain 2022 Tactical Analysis
